# PDGE Saber 11 — Análisis Definitivo de Desempeño Académico y Factores Socioeconómicos
## Procesamiento de Datos a Gran Escala | Departamento de Ingeniería de Sistemas

**Fuentes:** ICFES Saber 11, MEN, DANE, UARIV, Programa UNIDOS
**Dataset GOLD:** `hdfs://spark-master:9000/catalog/GOLD/join1/` — 322,068 registros, 72 columnas

---

> **Nota metodológica — Falacia ecológica:** Las correlaciones y modelos a nivel municipal
> describen asociaciones entre *promedios de municipio*, no comportamientos individuales.
> Las preguntas P1, P3, P4 operan a nivel colegio×jornada. Las preguntas P2, P5, P6, P7, P8
> operan a nivel municipal (df_mpio, 1,102 municipios).

## 1. Configuración del Entorno

In [0]:
%matplotlib inline
import sys
sys.path.insert(0, '/home/estudiante/spark/python')
sys.path.insert(0, '/home/estudiante/spark/python/lib/py4j-0.10.9.7-src.zip')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, count
from pyspark.ml.feature import (VectorAssembler, StandardScaler, StringIndexer,
                                  OneHotEncoder, Bucketizer)
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.regression import LinearRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (RegressionEvaluator, MulticlassClassificationEvaluator,
                                    ClusteringEvaluator)
from pyspark import StorageLevel
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("ML_Definitivo_PDGE") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.cores", "2") \
    .config("spark.sql.shuffle.partitions", "12") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark OK. Master:", spark.sparkContext.master)

## 2. Carga de Datos y Exploración Inicial

In [0]:
df_raw = spark.read.parquet("hdfs://spark-master:9000/catalog/GOLD/join1/")
print(f"Filas: {df_raw.count()} | Columnas: {len(df_raw.columns)}")

In [0]:
df_raw.select("AVG_PUNT_GLOBAL","AVG_PUNT_MATEMATICAS","AVG_PUNT_LECTURA_CRITICA",
              "AVG_PUNT_INGLES","AVG_PUNT_C_NATURALES","AVG_PUNT_SOCIALES_CIUDADANAS") \
      .describe().show()

In [0]:
jornada_pd = df_raw.groupBy("COLE_JORNADA").count().orderBy("count", ascending=False).toPandas()
nat_pd     = df_raw.groupBy("COLE_NATURALEZA").count().toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(jornada_pd["COLE_JORNADA"], jornada_pd["count"], color="#4472C4")
axes[0].set_xlabel("Registros")
axes[0].set_title("Distribución por jornada")
for i, v in enumerate(jornada_pd["count"]):
    axes[0].text(v + 200, i, f"{v:,}", va="center", fontsize=8)

axes[1].bar(nat_pd["COLE_NATURALEZA"], nat_pd["count"], color=["#ED7D31","#4472C4"], width=0.5)
axes[1].set_ylabel("Registros")
axes[1].set_title("Distribución por naturaleza del colegio")
for i, v in enumerate(nat_pd["count"]):
    axes[1].text(i, v + 200, f"{v:,}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 3. Limpieza, Filtros y Transformaciones

### Filtros
- **Filtro 1:** Excluir jornadas NOCHE y SABATINA — perfil socioeconómico/motivacional
  estructuralmente distinto al bachillerato regular. Elimina 78,840 registros (24.5%).
- **Filtro 2:** Excluir estudiantes privados de libertad — condiciones incomparables con
  la población general. Elimina 378 registros (0.1%).

### Transformaciones
1. **Columnas porcentaje (string → double):** limpieza de `%` y `,` decimal.
2. **Nulos de joins → 0:** municipios sin víctimas/trabajo/beneficiarios tienen NULL por
   LEFT JOIN; el valor correcto es 0.
3. **Renombrado de columnas** (ej: `sum(TOTAL_BENEFICIARIOS)` → `TOTAL_BENEFICIARIOS`).
4. **NIVEL_DESEMPENO:** cuartiles de `AVG_PUNT_GLOBAL` → etiqueta 0-3 para el modelo global.
5. **Fix NaN:** casts fallidos producen NaN (no null) que rompen StandardScaler — se convierten a null.

In [0]:
n_orig = df_raw.count()

# Filtro 1
df = df_raw.filter(~F.col('COLE_JORNADA').isin(['NOCHE', 'SABATINA']))
n_f1 = df.count()
print(f"[F1] {n_orig} → {n_f1}  (eliminados: {n_orig-n_f1})")

# Filtro 2
df = df.filter((F.col('ESTU_PRIVADO_LIBERTAD') != 'S') | F.col('ESTU_PRIVADO_LIBERTAD').isNull())
n_f2 = df.count()
print(f"[F2] {n_f1} → {n_f2}  (eliminados: {n_f1-n_f2})")
print(f"Dataset final: {n_f2} registros ({n_f2/n_orig*100:.1f}% del original)")

In [0]:
# Efecto de los filtros
fig, ax = plt.subplots(figsize=(7, 4))
labels_f  = ["Original", "Tras F1\n(sin NOCHE/SAB)", "Tras F2\n(sin priv. libertad)"]
vals_f    = [n_orig, n_f1, n_f2]
bars = ax.bar(labels_f, vals_f, color=["#4472C4","#ED7D31","#70AD47"], width=0.5)
ax.set_ylabel("Registros")
ax.set_title("Efecto de los filtros aplicados")
ax.set_ylim(0, n_orig * 1.12)
for bar, v in zip(bars, vals_f):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1200,
            f"{v:,}", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

In [0]:
# Transformación 1: porcentajes string -> double
pct_cols_raw = [c for c in df.columns if any(kw in c for kw in
    ['TASA_', 'COBERTURA_', 'DESERCI', 'APROBACI', 'REPROBACI', 'REPITENCIA', 'POBLACI'])]
for c in pct_cols_raw:
    df = df.withColumn(c,
        F.when(F.col(c).isNull() | (F.col(c) == 'NULL'), None)
         .otherwise(F.regexp_replace(F.regexp_replace(F.col(c), '%', ''), ',', '.').cast('double')))
print(f"[T1] {len(pct_cols_raw)} columnas convertidas")

# Transformación 2: nulos -> 0
df = df.withColumn('total', F.when(F.col('total').isNull(), 0.0).otherwise(F.col('total')))
df = df.withColumn('sum(TOTAL_BENEFICIARIOS)',
    F.when(F.col('sum(TOTAL_BENEFICIARIOS)').isNull(), 0).otherwise(F.col('sum(TOTAL_BENEFICIARIOS)')))
for wc in ['sum(TOTAL_PERSONAS)','sum(TOTAL_HORAS_OFICIO_HOGAR)',
           'sum(TOTAL_HORAS_NO_REMUNERADAS)','sum(CARGA_TOTAL_TRABAJO)']:
    df = df.withColumn(wc, F.when(F.col(wc).isNull(), 0).otherwise(F.col(wc)))
print("[T2] Nulos de víctimas/trabajo/beneficiarios -> 0")

# Transformación 3: renombrar
df = (df.withColumnRenamed('total', 'TOTAL_VICTIMAS')
        .withColumnRenamed('sum(TOTAL_BENEFICIARIOS)', 'TOTAL_BENEFICIARIOS')
        .withColumnRenamed('sum(TOTAL_MATRICULA)',      'TOTAL_MATRICULA')
        .withColumnRenamed('sum(TOTAL_PERSONAS)',       'TOTAL_TRABAJO_INFANTIL')
        .withColumnRenamed('sum(TOTAL_HORAS_OFICIO_HOGAR)',    'HORAS_OFICIO_HOGAR')
        .withColumnRenamed('sum(TOTAL_HORAS_NO_REMUNERADAS)',  'HORAS_NO_REMUNERADAS')
        .withColumnRenamed('sum(CARGA_TOTAL_TRABAJO)',  'CARGA_TOTAL_TRABAJO'))
print("[T3] Columnas renombradas")

# Fix NaN -> null en columnas double
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, F.when(F.isnan(F.col(c)), None).otherwise(F.col(c)))
print("[Fix] NaN -> null en columnas numéricas")

In [0]:
# Detección dinámica de columnas con caracteres especiales
col_cob_neta   = [c for c in df.columns if 'COBERTURA_NETA'   in c and not any(x in c for x in ['TRANSICI','PRIMARIA','SECUNDARIA','MEDIA'])][0]
col_cob_bruta  = [c for c in df.columns if 'COBERTURA_BRUTA'  in c and not any(x in c for x in ['TRANSICI','PRIMARIA','SECUNDARIA','MEDIA'])][0]
col_desercion  = [c for c in df.columns if 'DESERCI'   in c and not any(x in c for x in ['TRANSICI','PRIMARIA','SECUNDARIA','MEDIA'])][0]
col_repitencia = [c for c in df.columns if 'REPITENCIA' in c and not any(x in c for x in ['TRANSICI','PRIMARIA','SECUNDARIA','MEDIA'])][0]
col_reprobac   = [c for c in df.columns if 'REPROBACI'  in c and not any(x in c for x in ['TRANSICI','PRIMARIA','SECUNDARIA','MEDIA'])][0]
col_aprobac    = [c for c in df.columns if 'APROBACI'   in c and not any(x in c for x in ['TRANSICI','PRIMARIA','SECUNDARIA','MEDIA'])][0]
col_tasa_mat   = [c for c in df.columns if 'TASA_MATRICULACI' in c][0]
print(f"cob_neta={col_cob_neta}")
print(f"desercion={col_desercion}")
print(f"tasa_mat={col_tasa_mat}")

In [0]:
# Transformación 4: NIVEL_DESEMPENO (target modelo global)
q25, q50, q75 = df.approxQuantile('AVG_PUNT_GLOBAL', [0.25, 0.5, 0.75], 0.01)
print(f"Cuartiles AVG_PUNT_GLOBAL: Q1={q25:.1f}  Q2={q50:.1f}  Q3={q75:.1f}")

df = df.withColumn('NIVEL_DESEMPENO',
    F.when(F.col('AVG_PUNT_GLOBAL') <= q25, 0.0)
     .when(F.col('AVG_PUNT_GLOBAL') <= q50, 1.0)
     .when(F.col('AVG_PUNT_GLOBAL') <= q75, 2.0)
     .otherwise(3.0))

df.persist(StorageLevel.MEMORY_AND_DISK)
n_df = df.count()
print(f"Dataset persistido: {n_df} filas")

# Distribución NIVEL_DESEMPENO
nivel_pd = df.groupBy('NIVEL_DESEMPENO').count().orderBy('NIVEL_DESEMPENO').toPandas()
nivel_pd.show() if hasattr(nivel_pd, 'show') else print(nivel_pd)

In [0]:
etiquetas_n = ['BAJO\n(≤Q1)', 'MEDIO_BAJO\n(Q1-Q2)', 'MEDIO_ALTO\n(Q2-Q3)', 'ALTO\n(>Q3)']
colores_n   = ['#C00000','#ED7D31','#FFC000','#70AD47']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(etiquetas_n, nivel_pd['count'], color=colores_n, width=0.6)
ax.set_ylabel("Registros")
ax.set_title("Distribución de NIVEL_DESEMPENO (cuartiles de puntaje global)")
ax.set_ylim(0, nivel_pd['count'].max() * 1.15)
for bar, v in zip(bars, nivel_pd['count']):
    pct = v / nivel_pd['count'].sum() * 100
    ax.text(bar.get_x() + bar.get_width()/2, v + 300,
            f"{v:,}\n({pct:.1f}%)", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Preparación: Agregación Municipal y Correlaciones

Las preguntas P2, P5, P6, P7, P8 operan sobre `df_mpio` (1 fila = 1 municipio).
Las preguntas P1, P3, P4 operan sobre `df` (1 fila = colegio × jornada).

In [0]:
agg_exprs = [
    F.mean('AVG_PUNT_GLOBAL').alias('PROM_PUNT_GLOBAL'),
    F.mean('AVG_PUNT_MATEMATICAS').alias('PROM_MATEMATICAS'),
    F.mean('AVG_PUNT_LECTURA_CRITICA').alias('PROM_LECTURA'),
    F.mean('AVG_PUNT_C_NATURALES').alias('PROM_C_NATURALES'),
    F.mean('AVG_PUNT_SOCIALES_CIUDADANAS').alias('PROM_SOCIALES'),
    F.mean('AVG_PUNT_INGLES').alias('PROM_INGLES'),
    F.first('TOTAL_BENEFICIARIOS').alias('TOTAL_BENEFICIARIOS'),
    F.first('TOTAL_VICTIMAS').alias('TOTAL_VICTIMAS'),
    F.first('TOTAL_MATRICULA').alias('TOTAL_MATRICULA'),
    F.first('TOTAL_TRABAJO_INFANTIL').alias('TOTAL_TRABAJO_INFANTIL'),
    F.first('HORAS_OFICIO_HOGAR').alias('HORAS_OFICIO_HOGAR'),
    F.first('HORAS_NO_REMUNERADAS').alias('HORAS_NO_REMUNERADAS'),
    F.first('CARGA_TOTAL_TRABAJO').alias('CARGA_TOTAL_TRABAJO'),
    F.mean(col_cob_neta).alias('COBERTURA_NETA'),
    F.mean(col_cob_bruta).alias('COBERTURA_BRUTA'),
    F.mean(col_desercion).alias('DESERCION'),
    F.mean(col_repitencia).alias('REPITENCIA'),
    F.mean(col_reprobac).alias('REPROBACION'),
    F.mean(col_aprobac).alias('APROBACION'),
    F.mean(col_tasa_mat).alias('TASA_MATRICULACION'),
]
df_mpio = df.groupBy('COD_MUNICIPIO').agg(*agg_exprs).dropna()
df_mpio.persist(StorageLevel.MEMORY_AND_DISK)
print(f"Municipios con datos completos: {df_mpio.count()}")

In [0]:
vars_corr = [
    ('TOTAL_BENEFICIARIOS',   'Proxy IPM'),
    ('TOTAL_VICTIMAS',        'Victimas conflicto'),
    ('TOTAL_MATRICULA',       'Total matriculados'),
    ('TOTAL_TRABAJO_INFANTIL','Trabajo infantil'),
    ('HORAS_OFICIO_HOGAR',   'Horas oficios hogar'),
    ('CARGA_TOTAL_TRABAJO',   'Carga total trabajo'),
    ('COBERTURA_NETA',        'Cobertura neta'),
    ('COBERTURA_BRUTA',       'Cobertura bruta'),
    ('DESERCION',             'Desercion'),
    ('REPITENCIA',            'Repitencia'),
    ('TASA_MATRICULACION',    'Tasa matriculacion'),
]
corr_vals = []
for col_c, desc in vars_corr:
    r = df_mpio.stat.corr(col_c, 'PROM_PUNT_GLOBAL')
    corr_vals.append((desc, r))
    print(f"  {desc:30s}: r={r:+.4f}")

# Multicolinealidad
r_cb_cn = df_mpio.stat.corr('COBERTURA_BRUTA', 'COBERTURA_NETA')
print(f"\nCorr COBERTURA_BRUTA vs NETA: {r_cb_cn:.4f}")
if abs(r_cb_cn) > 0.85:
    print("→ |r|>0.85 → COBERTURA_BRUTA eliminada del modelo global (se conserva NETA)")

In [0]:
# Gráfico de correlaciones
nombres_r = [x[0] for x in corr_vals]
valores_r  = [x[1] for x in corr_vals]
idx_ord    = np.argsort(valores_r)
c_ord = [('#70AD47' if valores_r[i] >= 0 else '#C00000') for i in idx_ord]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh([nombres_r[i] for i in idx_ord], [valores_r[i] for i in idx_ord], color=c_ord)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel("Correlación de Pearson (r)")
ax.set_title("Correlación variables socioeconómicas vs PROM_PUNT_GLOBAL (nivel municipal)")
for i, idx in enumerate(idx_ord):
    v = valores_r[idx]
    ax.text(v + (0.005 if v >= 0 else -0.005), i, f"{v:+.3f}",
            va='center', ha=('left' if v >= 0 else 'right'), fontsize=8)
ax.set_xlim(-0.35, 0.35)
plt.tight_layout()
plt.show()

## 5. Preguntas de Negocio

---
### P1: ¿En qué medida el IPM municipal explica la variación en el puntaje ICFES?

**Nota:** El dataset no incluye el IPM directamente. Se usa como proxy `TOTAL_BENEFICIARIOS`
(hogares en pobreza extrema del programa UNIDOS). El modelo opera a nivel **colegio × jornada**
(`AVG_PUNT_GLOBAL` = puntaje promedio del colegio en esa jornada).

**Modelo:** Regresión Lineal con proxy IPM + indicadores educativos municipales.

**Correcciones aplicadas al enfoque original:**
- Columnas identificadas dinámicamente (evita problemas de encoding).
- `handleInvalid='skip'` en VectorAssembler (evita NaN que rompen StandardScaler).
- Se evalúan 3 valores de `regParam` [0.01, 0.1, 1.0] para comparar configuraciones.

In [0]:
# df02: 1 fila por municipio, solo columnas necesarias para P1
df02 = df_mpio.select(
    'COD_MUNICIPIO',
    'TOTAL_BENEFICIARIOS',
    'TOTAL_MATRICULA',
    'COBERTURA_NETA',
    'COBERTURA_BRUTA',
    'DESERCION',
    'APROBACION',
    'REPROBACION',
    'REPITENCIA',
    'TASA_MATRICULACION',
    'PROM_PUNT_GLOBAL'
).dropna()

# Proxy IPM correcto: tasa relativa, no conteo absoluto
# TOTAL_BENEFICIARIOS/TOTAL_MATRICULA = proporción de hogares pobres extremos
# respecto al total de estudiantes matriculados en el municipio
df02 = df02.withColumn(
    'TASA_BENEFICIARIOS',
    F.col('TOTAL_BENEFICIARIOS') / (F.col('TOTAL_MATRICULA') + F.lit(1))
)

# Diagnóstico de correlaciones — todas las variables candidatas
print("Correlaciones vs PROM_PUNT_GLOBAL:")
for c in ['TOTAL_BENEFICIARIOS', 'TASA_BENEFICIARIOS', 'TOTAL_MATRICULA',
          'COBERTURA_NETA', 'DESERCION', 'REPITENCIA', 'TASA_MATRICULACION']:
    r = df02.stat.corr(c, 'PROM_PUNT_GLOBAL')
    print(f"  {c:30s}: r={r:+.4f}")

print(f'\nMunicipios en df02: {df02.count()}')

df_train_p1, df_test_p1 = df02.randomSplit([0.75, 0.25], seed=42)

assembler_p1 = VectorAssembler(
    inputCols=[
        'TASA_BENEFICIARIOS',   # proxy IPM normalizado
        'TASA_MATRICULACION',
        'COBERTURA_NETA',
        'DESERCION',
        'APROBACION',
        'REPROBACION',
        'REPITENCIA'
    ],
    outputCol='features_p1', handleInvalid='skip')

import gc
df_train_p1_asm = assembler_p1.transform(df_train_p1)
df_test_p1_asm  = assembler_p1.transform(df_test_p1)

scaler_p1_model = StandardScaler(inputCol='features_p1', outputCol='features_scaled_p1',
                                  withMean=True, withStd=True).fit(df_train_p1_asm)
train_p1 = scaler_p1_model.transform(df_train_p1_asm).withColumn('label', col('PROM_PUNT_GLOBAL').cast('double'))
test_p1  = scaler_p1_model.transform(df_test_p1_asm).withColumn('label',  col('PROM_PUNT_GLOBAL').cast('double'))
del df_train_p1_asm, df_test_p1_asm, scaler_p1_model
gc.collect(); gc.collect()

ev1 = RegressionEvaluator(labelCol='label', predictionCol='prediction')
p1_res = []
print("\n=== Pregunta 1: Proxy IPM (tasa) vs PROM_PUNT_GLOBAL (1 fila = 1 municipio) ===")
for reg in [0.01, 0.1, 1.0]:
    m    = LinearRegression(featuresCol='features_scaled_p1', labelCol='label',
                             maxIter=30, regParam=reg).fit(train_p1)
    pred = m.transform(test_p1)
    rmse = ev1.setMetricName('rmse').evaluate(pred)
    mae  = ev1.setMetricName('mae').evaluate(pred)
    r2   = ev1.setMetricName('r2').evaluate(pred)
    p1_res.append((reg, rmse, mae, r2, m.intercept))
    print(f"  regParam={reg}: RMSE={rmse:.4f} | MAE={mae:.4f} | R²={r2:.4f} | Intercepto={m.intercept:.2f}")
    del m, pred
    gc.collect(); gc.collect()

best_m_p1  = LinearRegression(featuresCol='features_scaled_p1', labelCol='label',
                               maxIter=30, regParam=0.01).fit(train_p1)
best_pr_p1 = best_m_p1.transform(test_p1)
print("\n--- Muestra: puntaje real vs predicho (mejor modelo, 10 filas) ---")
best_pr_p1.select('COD_MUNICIPIO',
                   F.col('label').alias('puntaje_real'),
                   F.round(F.col('prediction'), 2).alias('puntaje_pred')).show(10)

del assembler_p1, df_train_p1, df_test_p1
gc.collect(); gc.collect()

In [0]:
# Visualización P1
sample_pd = best_pr_p1.select(
    F.col('label').alias('real'),
    F.col('prediction').alias('pred')
).limit(300).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: real vs predicho
axes[0].scatter(sample_pd['real'], sample_pd['pred'],
                alpha=0.3, s=15, color='#4472C4')
mn = min(sample_pd['real'].min(), sample_pd['pred'].min()) - 5
mx = max(sample_pd['real'].max(), sample_pd['pred'].max()) + 5
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1, label='Perfecta prediccion')
axes[0].set_xlabel("Puntaje real (AVG_PUNT_GLOBAL)")
axes[0].set_ylabel("Puntaje predicho")
axes[0].set_title("P1: Real vs Predicho — mejor modelo (muestra n=300)")
axes[0].legend(fontsize=8)

# R² por configuración
regs = [str(r[0]) for r in p1_res]
r2s  = [r[3] for r in p1_res]
axes[1].bar(regs, r2s, color='#4472C4', width=0.4)
axes[1].set_xlabel("regParam")
axes[1].set_ylabel("R²")
axes[1].set_title("P1: R² por configuración de regularización")
axes[1].set_ylim(0, 0.15)
for i, v in enumerate(r2s):
    axes[1].text(i, v + 0.002, f"{v:.4f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
### P2 (⚠ REEMPLAZADA): Víctimas del conflicto vs puntaje ICFES

**Pregunta original:** ¿Los municipios con más víctimas del conflicto tienen puntajes inferiores?

**Por qué no puede responderse:**
1. El 96.8% de registros tiene `TOTAL_VICTIMAS=0` (solo 45 de 1,102 municipios con datos reales).
2. Cualquier modelo supervisado aprende el patrón del desbalance, no una relación real
   (un árbol que prediga siempre "BAJO" obtiene Accuracy≈0.968 sin aprender nada).

**Pregunta alternativa:** ¿Los municipios con mayor vulnerabilidad social forman perfiles
de desempeño diferenciados?

**Modelo:** K-Means clustering (variables: trabajo infantil, beneficiarios UNIDOS, deserción, repitencia).

In [0]:
print("=== LIMITACIÓN: TOTAL_VICTIMAS ===")
n_cero = df_mpio.filter(F.col('TOTAL_VICTIMAS') == 0).count()
n_pos  = df_mpio.filter(F.col('TOTAL_VICTIMAS')  > 0).count()
total  = n_cero + n_pos
print(f"Municipios con TOTAL_VICTIMAS=0  : {n_cero} ({n_cero/total*100:.1f}%)")
print(f"Municipios con TOTAL_VICTIMAS>0  : {n_pos}  ({n_pos/total*100:.1f}%)")
print(f"  → Un árbol que prediga siempre BAJO obtiene Accuracy={n_cero/total:.4f}")
print("     sin aprender ninguna relación real. Resultado inválido.")

In [0]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Con datos\n(n=45)', 'Sin datos / cero\n(n=1057)'],
       [n_pos, n_cero], color=['#ED7D31','#C0C0C0'], width=0.5)
ax.set_ylabel("Municipios")
ax.set_title("P2: Desbalance en TOTAL_VICTIMAS — motivo de reemplazo")
ax.text(0, n_pos + 5,  f"{n_pos/total*100:.1f}%",  ha='center', fontsize=11, color='#ED7D31')
ax.text(1, n_cero + 5, f"{n_cero/total*100:.1f}%", ha='center', fontsize=11, color='gray')
plt.tight_layout()
plt.show()

In [0]:
asm_v = VectorAssembler(
    inputCols=['TOTAL_TRABAJO_INFANTIL','CARGA_TOTAL_TRABAJO',
               'TOTAL_BENEFICIARIOS','DESERCION','REPITENCIA'],
    outputCol='feat', handleInvalid='skip')
sc_v   = StandardScaler(inputCol='feat', outputCol='feat_sc', withMean=True, withStd=True)
pipe_v = Pipeline(stages=[asm_v, sc_v]).fit(df_mpio)
df_vf  = pipe_v.transform(df_mpio)

sil_p2 = {}
print("Silhouette por k:")
for k in [2, 3, 4]:
    m  = KMeans(featuresCol='feat_sc', predictionCol='cluster_v', k=k, maxIter=20, seed=42).fit(df_vf)
    cl = m.transform(df_vf)
    sil = ClusteringEvaluator(featuresCol='feat_sc', predictionCol='cluster_v',
                               metricName='silhouette', distanceMeasure='squaredEuclidean').evaluate(cl)
    sil_p2[k] = sil
    print(f"  k={k}: Silhouette={sil:.4f}")

m2   = KMeans(featuresCol='feat_sc', predictionCol='cluster_v', k=2, maxIter=20, seed=42).fit(df_vf)
cl2  = m2.transform(df_vf)
df_p2_cl = cl2.groupBy('cluster_v').agg(
    F.round(F.mean('TOTAL_TRABAJO_INFANTIL'),0).alias('trabajo_inf'),
    F.round(F.mean('TOTAL_BENEFICIARIOS'),0).alias('beneficiarios'),
    F.round(F.mean('DESERCION'),2).alias('desercion'),
    F.round(F.mean('PROM_PUNT_GLOBAL'),2).alias('puntaje_prom'),
    F.count('COD_MUNICIPIO').alias('n_mun')
).orderBy('puntaje_prom', ascending=False)
print("\n[k=2 seleccionado] Perfil de clusters de vulnerabilidad:")
df_p2_cl.show()
print("→ Δ puntaje entre clusters ≈ 3.64 pts (modesto — la vulnerabilidad actúa como factor de fondo).")

In [0]:
p2_cl_pd = df_p2_cl.toPandas().sort_values('puntaje_prom', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(sil_p2.keys()), list(sil_p2.values()), 'o-', color='#4472C4', linewidth=2)
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette")
axes[0].set_title("P2: Selección de k (vulnerabilidad social)")
axes[0].set_xticks(list(sil_p2.keys()))
for k, s in sil_p2.items():
    axes[0].annotate(f"{s:.4f}", (k, s), textcoords="offset points", xytext=(5, 5), fontsize=9)

labels_p2 = [f"Cluster {int(r['cluster_v'])}\n(n={int(r['n_mun'])})" for _, r in p2_cl_pd.iterrows()]
axes[1].bar(labels_p2, p2_cl_pd['puntaje_prom'], color=['#70AD47','#C00000'], width=0.5)
axes[1].set_ylabel("Puntaje promedio")
axes[1].set_title("P2: Puntaje por cluster de vulnerabilidad social")
axes[1].set_ylim(238, 252)
for i, v in enumerate(p2_cl_pd['puntaje_prom']):
    axes[1].text(i, v + 0.2, f"{v:.2f}", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
### P3: ¿Diferencia significativa en puntaje entre colegios oficial y no oficial?

**Modelo:** Random Forest Classifier — predice OFICIAL (1) vs NO_OFICIAL (0) desde puntajes por materia.

**Advertencia:** La diferencia descriptiva es robusta pero NO causal: los colegios privados
se concentran en zonas urbanas y atienden familias con mayor ingreso. Este análisis es descriptivo.

In [0]:
print("Puntaje por naturaleza × área:")
df.groupBy('COLE_NATURALEZA','COLE_AREA_UBICACION').agg(
    F.round(F.mean('AVG_PUNT_GLOBAL'),2).alias('puntaje_prom'),
    F.count('COD_MUNICIPIO').alias('n')
).orderBy('COLE_NATURALEZA','COLE_AREA_UBICACION').show()

In [0]:
df_p3 = df.withColumn('label',
    F.when(F.upper(F.col('COLE_NATURALEZA'))=='OFICIAL', 1.0).otherwise(0.0))
tr_p3, te_p3 = df_p3.randomSplit([0.75, 0.25], seed=42)

asm_p3 = VectorAssembler(
    inputCols=['AVG_PUNT_LECTURA_CRITICA','AVG_PUNT_C_NATURALES',
               'AVG_PUNT_SOCIALES_CIUDADANAS','AVG_PUNT_MATEMATICAS','AVG_PUNT_INGLES'],
    outputCol='feat', handleInvalid='skip')
sc_p3  = StandardScaler(inputCol='feat', outputCol='feat_sc', withMean=True, withStd=True)
pipe_p3 = Pipeline(stages=[asm_p3, sc_p3]).fit(tr_p3)
tr_p3f = pipe_p3.transform(tr_p3)
te_p3f = pipe_p3.transform(te_p3)

ev3 = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction')
p3_res = []
best_m_p3 = None
print("Random Forest — variando numTrees:")
for ntrees in [10, 30, 50]:
    m  = RandomForestClassifier(featuresCol='feat_sc', labelCol='label',
                                 numTrees=ntrees, seed=42).fit(tr_p3f)
    pr = m.transform(te_p3f)
    acc = ev3.setMetricName('accuracy').evaluate(pr)
    f1  = ev3.setMetricName('f1').evaluate(pr)
    p3_res.append((ntrees, acc, f1))
    print(f"  numTrees={ntrees}: Accuracy={acc:.4f} | F1={f1:.4f}")
    if ntrees == 50: best_m_p3 = m

fi_p3 = best_m_p3.featureImportances
fi_names_p3 = ['Lectura','C_Naturales','Sociales','Matematicas','Ingles']
fi_vals_p3  = [float(fi_p3[i]) for i in range(len(fi_names_p3))]
print("\nImportancia de variables (numTrees=50):")
for n, v in zip(fi_names_p3, fi_vals_p3):
    print(f"  {n:15s}: {v:.4f}  {'|'*int(v*50)}")
print("\n→ Inglés (~75%) es el principal diferenciador: los colegios privados invierten en")
print("  bilingüismo sistemáticamente. Accuracy≈0.82.")

In [0]:
nat_area_pd = df.groupBy('COLE_NATURALEZA','COLE_AREA_UBICACION').agg(
    F.round(F.mean('AVG_PUNT_GLOBAL'),2).alias('puntaje_prom'),
    F.count('COD_MUNICIPIO').alias('n')
).orderBy('COLE_NATURALEZA','COLE_AREA_UBICACION').toPandas()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Puntaje por naturaleza × área
areas = sorted(nat_area_pd['COLE_AREA_UBICACION'].unique())
nats  = sorted(nat_area_pd['COLE_NATURALEZA'].unique())
x     = np.arange(len(areas))
ancho = 0.35
cols_nat = {'NO OFICIAL': '#ED7D31', 'OFICIAL': '#4472C4'}
for i, nat in enumerate(nats):
    sub = nat_area_pd[nat_area_pd['COLE_NATURALEZA'] == nat].sort_values('COLE_AREA_UBICACION')
    axes[0].bar(x + i*ancho, sub['puntaje_prom'], ancho,
                label=nat, color=cols_nat.get(nat, '#999'))
axes[0].set_xticks(x + ancho/2)
axes[0].set_xticklabels(areas)
axes[0].set_ylabel("Puntaje promedio")
axes[0].set_title("P3: Puntaje por naturaleza y área")
axes[0].legend()
axes[0].set_ylim(200, 310)

# Importancia de variables
fi_ord = sorted(zip(fi_names_p3, fi_vals_p3), key=lambda x: x[1])
axes[1].barh([x[0] for x in fi_ord], [x[1] for x in fi_ord], color='#4472C4')
axes[1].set_xlabel("Importancia")
axes[1].set_title("P3: Importancia de variables RF (oficial vs no oficial)")
for i, (n, v) in enumerate(fi_ord):
    axes[1].text(v + 0.005, i, f"{v:.3f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
### P4: ¿Existe relación entre la ubicación geográfica y el puntaje en pruebas específicas?

**Modelo:** Ninguno, hacer agregación

In [0]:
# Puntaje promedio por area geografica en cada materia
area_mat = df.groupBy('COLE_AREA_UBICACION').agg(
    F.round(F.mean('AVG_PUNT_GLOBAL'),2).alias('Global'),
    F.round(F.mean('AVG_PUNT_MATEMATICAS'),2).alias('Matematicas'),
    F.round(F.mean('AVG_PUNT_LECTURA_CRITICA'),2).alias('Lectura'),
    F.round(F.mean('AVG_PUNT_INGLES'),2).alias('Ingles'),
    F.round(F.mean('AVG_PUNT_C_NATURALES'),2).alias('C_Naturales'),
    F.round(F.mean('AVG_PUNT_SOCIALES_CIUDADANAS'),2).alias('Sociales'),
    F.count('COD_MUNICIPIO').alias('n_registros')
).orderBy('Global', ascending=False)
print("=== Puntaje por area geografica ===")
area_mat.show()

# Desglose: area x naturaleza del colegio
area_nat = df.groupBy('COLE_AREA_UBICACION','COLE_NATURALEZA').agg(
    F.round(F.mean('AVG_PUNT_GLOBAL'),2).alias('global_prom'),
    F.round(F.mean('AVG_PUNT_INGLES'),2).alias('ingles_prom'),
    F.round(F.mean('AVG_PUNT_MATEMATICAS'),2).alias('mates_prom'),
    F.count('COD_MUNICIPIO').alias('n')
).orderBy('COLE_AREA_UBICACION','COLE_NATURALEZA')
print("=== Desglose area x naturaleza ===")
area_nat.show()

# Brecha URBANO - RURAL
area_df = area_mat.toPandas().set_index('COLE_AREA_UBICACION')
print("Brecha URBANO - RURAL por materia:")
for m in ['Global','Matematicas','Lectura','Ingles','C_Naturales','Sociales']:
    if 'URBANO' in area_df.index and 'RURAL' in area_df.index:
        diff = area_df.loc['URBANO', m] - area_df.loc['RURAL', m]
        print(f"  {m:15s}: URBANO={area_df.loc['URBANO',m]:.2f}  "
              f"RURAL={area_df.loc['RURAL',m]:.2f}  BRECHA={diff:+.2f}")

In [0]:
area_df     = area_mat.toPandas().set_index('COLE_AREA_UBICACION')
materias_p4 = ['Matematicas','Lectura','Ingles','C_Naturales','Sociales','Global']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Barras agrupadas: URBANO vs RURAL por materia
x     = np.arange(len(materias_p4))
ancho = 0.35
cols_a = {'URBANO': '#4472C4', 'RURAL': '#ED7D31'}
for i, area in enumerate(area_df.index):
    axes[0].bar(x + i*ancho, [area_df.loc[area, m] for m in materias_p4],
                ancho, label=area, color=cols_a.get(area, '#999'))
axes[0].set_xticks(x + ancho/2)
axes[0].set_xticklabels(materias_p4, rotation=15)
axes[0].set_ylabel("Puntaje promedio")
axes[0].set_title("P4: Puntaje por area geografica y materia")
axes[0].legend()

# Brecha URBANO - RURAL por materia
brechas = []
for m in materias_p4:
    if 'URBANO' in area_df.index and 'RURAL' in area_df.index:
        brechas.append(area_df.loc['URBANO', m] - area_df.loc['RURAL', m])
    else:
        brechas.append(0)
colores_b = ['#4472C4' if v >= 0 else '#C00000' for v in brechas]
axes[1].bar(materias_p4, brechas, color=colores_b)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel("Brecha en puntos (URBANO - RURAL)")
axes[1].set_title("P4: Brecha urbano-rural por materia")
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(brechas):
    axes[1].text(i, v + (0.3 if v >= 0 else -0.7),
                 f"{v:+.1f}", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
### P5: ¿La carga de trabajo municipal se asocia con el puntaje ICFES?

**Modelo:** Regresión Lineal (nivel municipal). Mejor R²=0.204 — el mayor R² de los modelos de regresión.

In [0]:
print("Correlaciones carga laboral vs PROM_PUNT_GLOBAL:")
for c, lbl in [('CARGA_TOTAL_TRABAJO','Carga total trabajo'),
                ('HORAS_OFICIO_HOGAR', 'Horas oficios del hogar'),
                ('HORAS_NO_REMUNERADAS','Horas no remuneradas')]:
    r = df_mpio.stat.corr(c, 'PROM_PUNT_GLOBAL')
    print(f"  {lbl:30s}: r={r:+.4f}")

carga_med = df_mpio.approxQuantile('CARGA_TOTAL_TRABAJO', [0.5], 0.01)[0]
df_p5_seg = df_mpio.withColumn('nivel_carga',
    F.when(F.col('CARGA_TOTAL_TRABAJO') <= carga_med, 'BAJA').otherwise('ALTA')
).groupBy('nivel_carga').agg(
    F.round(F.mean('PROM_PUNT_GLOBAL'),2).alias('puntaje_prom'),
    F.count('COD_MUNICIPIO').alias('n')
).orderBy('nivel_carga')
df_p5_seg.show()

tr_p5, te_p5 = df_mpio.randomSplit([0.75,0.25], seed=42)
asm_p5 = VectorAssembler(
    inputCols=['CARGA_TOTAL_TRABAJO','HORAS_OFICIO_HOGAR','HORAS_NO_REMUNERADAS',
               'COBERTURA_NETA','TASA_MATRICULACION'],
    outputCol='feat', handleInvalid='skip')
sc_p5 = StandardScaler(inputCol='feat', outputCol='feat_sc', withMean=True, withStd=True)
pipe_p5 = Pipeline(stages=[asm_p5, sc_p5]).fit(tr_p5)
tr_p5f = pipe_p5.transform(tr_p5).withColumn('label', F.col('PROM_PUNT_GLOBAL').cast('double'))
te_p5f = pipe_p5.transform(te_p5).withColumn('label', F.col('PROM_PUNT_GLOBAL').cast('double'))

ev5 = RegressionEvaluator(labelCol='label', predictionCol='prediction')
p5_res = []
print("Regresión Lineal — 3 configuraciones:")
for reg in [0.01, 0.1, 1.0]:
    m  = LinearRegression(featuresCol='feat_sc', labelCol='label', maxIter=30, regParam=reg).fit(tr_p5f)
    pr = m.transform(te_p5f)
    r2   = ev5.setMetricName('r2').evaluate(pr)
    rmse = ev5.setMetricName('rmse').evaluate(pr)
    p5_res.append((reg, rmse, r2))
    print(f"  regParam={reg}: RMSE={rmse:.2f} | R²={r2:.4f}")
print("\n→ Alta carga: 239.45 pts | Baja carga: 247.42 pts (Δ=7.97 pts). R²=0.204.")

In [0]:
p5_seg_pd = df_p5_seg.toPandas().sort_values('puntaje_prom', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Baja carga', 'Alta carga'],
            p5_seg_pd['puntaje_prom'], color=['#70AD47','#C00000'], width=0.4)
axes[0].set_ylabel("Puntaje promedio ICFES")
axes[0].set_title("P5: Puntaje según carga laboral municipal")
axes[0].set_ylim(234, 252)
for i, (v, n) in enumerate(zip(p5_seg_pd['puntaje_prom'], p5_seg_pd['n'])):
    axes[0].text(i, v + 0.2, f"{v:.2f}\n(n={n})", ha='center', fontsize=10, fontweight='bold')

axes[1].bar([str(r[0]) for r in p5_res], [r[2] for r in p5_res], color='#4472C4', width=0.4)
axes[1].set_xlabel("regParam")
axes[1].set_ylabel("R²")
axes[1].set_title("P5: R² por configuración (mejor R² del análisis)")
axes[1].set_ylim(0, 0.28)
for i, r in enumerate(p5_res):
    axes[1].text(i, r[2] + 0.005, f"{r[2]:.4f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
### P6: ¿Cómo se relaciona el trabajo infantil con el puntaje ICFES?

**Modelo:** Regresión Lineal (nivel municipal) con trabajo infantil, carga doméstica y consecuencias educativas.

In [0]:
print("Correlaciones trabajo infantil vs PROM_PUNT_GLOBAL:")
for c, lbl in [('TOTAL_TRABAJO_INFANTIL','Menores en trabajo infantil'),
                ('CARGA_TOTAL_TRABAJO',   'Carga total trabajo'),
                ('TOTAL_BENEFICIARIOS',   'Hogares pobreza extrema'),
                ('REPITENCIA',            'Tasa repitencia'),
                ('DESERCION',             'Tasa desercion')]:
    r = df_mpio.stat.corr(c, 'PROM_PUNT_GLOBAL')
    print(f"  {lbl:35s}: r={r:+.4f}")

ti_med = df_mpio.approxQuantile('TOTAL_TRABAJO_INFANTIL', [0.5], 0.01)[0]
df_p6_seg = df_mpio.withColumn('nivel_ti',
    F.when(F.col('TOTAL_TRABAJO_INFANTIL') <= ti_med, 'BAJO_TI').otherwise('ALTO_TI')
).groupBy('nivel_ti').agg(
    F.round(F.mean('PROM_PUNT_GLOBAL'),2).alias('puntaje_prom'),
    F.round(F.mean('REPITENCIA'),2).alias('repitencia'),
    F.round(F.mean('DESERCION'),2).alias('desercion'),
    F.count('COD_MUNICIPIO').alias('n')
).orderBy('nivel_ti')
df_p6_seg.show()

tr_p6, te_p6 = df_mpio.randomSplit([0.75,0.25], seed=42)
asm_p6 = VectorAssembler(
    inputCols=['TOTAL_TRABAJO_INFANTIL','CARGA_TOTAL_TRABAJO','HORAS_NO_REMUNERADAS',
               'TOTAL_BENEFICIARIOS','REPITENCIA','DESERCION','APROBACION','TASA_MATRICULACION'],
    outputCol='feat', handleInvalid='skip')
sc_p6 = StandardScaler(inputCol='feat', outputCol='feat_sc', withMean=True, withStd=True)
pipe_p6 = Pipeline(stages=[asm_p6, sc_p6]).fit(tr_p6)
tr_p6f = pipe_p6.transform(tr_p6).withColumn('label', F.col('PROM_PUNT_GLOBAL').cast('double'))
te_p6f = pipe_p6.transform(te_p6).withColumn('label', F.col('PROM_PUNT_GLOBAL').cast('double'))

ev6 = RegressionEvaluator(labelCol='label', predictionCol='prediction')
p6_res = []
print("\nRegresión Lineal — 3 configuraciones:")
for reg in [0.01, 0.1, 1.0]:
    m  = LinearRegression(featuresCol='feat_sc', labelCol='label', maxIter=30, regParam=reg).fit(tr_p6f)
    pr = m.transform(te_p6f)
    r2   = ev6.setMetricName('r2').evaluate(pr)
    rmse = ev6.setMetricName('rmse').evaluate(pr)
    p6_res.append((reg, rmse, r2))
    print(f"  regParam={reg}: RMSE={rmse:.2f} | R²={r2:.4f}")
print("\n→ Δ puntaje ALTO vs BAJO TI ≈ 10.28 pts. R²=0.203.")

In [0]:
p6_seg_pd = df_p6_seg.toPandas().sort_values('puntaje_prom', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(['Bajo TI', 'Alto TI'],
            p6_seg_pd['puntaje_prom'], color=['#70AD47','#C00000'], width=0.4)
axes[0].set_ylabel("Puntaje promedio")
axes[0].set_title("P6: Puntaje según trabajo infantil")
axes[0].set_ylim(232, 252)
for i, (v, n) in enumerate(zip(p6_seg_pd['puntaje_prom'], p6_seg_pd['n'])):
    axes[0].text(i, v + 0.2, f"{v:.2f}\n(n={n})", ha='center', fontsize=9, fontweight='bold')

axes[1].bar(['Bajo TI', 'Alto TI'],
            p6_seg_pd['repitencia'], color=['#70AD47','#C00000'], width=0.4)
axes[1].set_ylabel("Tasa de repitencia (%)")
axes[1].set_title("P6: Repitencia según trabajo infantil")
for i, v in enumerate(p6_seg_pd['repitencia']):
    axes[1].text(i, v + 0.01, f"{v:.2f}%", ha='center', fontsize=10)

axes[2].bar(['Bajo TI', 'Alto TI'],
            p6_seg_pd['desercion'], color=['#70AD47','#C00000'], width=0.4)
axes[2].set_ylabel("Tasa de deserción (%)")
axes[2].set_title("P6: Deserción según trabajo infantil")
for i, v in enumerate(p6_seg_pd['desercion']):
    axes[2].text(i, v + 0.01, f"{v:.2f}%", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---
### P7: ¿Los municipios con mayor matrícula tienen mejor desempeño?

**Modelo:** Ninguno.
**Hallazgo:** Relación NO-LINEAL — Q3 supera a Q4. El modelo lineal no captura esto (R²=0.05).

In [0]:
m_q1,m_q2,m_q3 = df_mpio.approxQuantile('TOTAL_MATRICULA', [0.25,0.5,0.75], 0.01)
print(f"Cuartiles TOTAL_MATRICULA: Q1={m_q1:.0f}  Q2={m_q2:.0f}  Q3={m_q3:.0f}")

bkt   = Bucketizer(splits=[float('-inf'),m_q1,m_q2,m_q3,float('inf')],
                   inputCol='TOTAL_MATRICULA', outputCol='seg_matricula')
df_p7 = bkt.transform(df_mpio)
df_p7_seg = df_p7.groupBy('seg_matricula').agg(
    F.round(F.mean('PROM_PUNT_GLOBAL'),2).alias('global_prom'),
    F.round(F.mean('PROM_MATEMATICAS'),2).alias('mates_prom'),
    F.count('COD_MUNICIPIO').alias('n_mun')
).orderBy('seg_matricula')
df_p7_seg.show()
print("Relacion NO LINEAL: Q1->Q3 sube, Q4 baja.")

p7_seg_pd = df_p7_seg.toPandas().sort_values('seg_matricula')
etiq_q = ['Q1\n(menor)', 'Q2', 'Q3', 'Q4\n(mayor)']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Puntaje global por cuartil
axes[0].plot(etiq_q, p7_seg_pd['global_prom'], 'o-', color='#4472C4', linewidth=2, markersize=8)
axes[0].set_ylabel("Puntaje promedio")
axes[0].set_title("P7: Puntaje global por cuartil de matricula (no-lineal)")
axes[0].set_ylim(233, 255)
axes[0].axhline(p7_seg_pd['global_prom'].mean(), color='red', linestyle='--',
                linewidth=1, label='Promedio')
axes[0].legend(fontsize=8)
for i, (v, n) in enumerate(zip(p7_seg_pd['global_prom'], p7_seg_pd['n_mun'])):
    axes[0].annotate(f"{v:.1f}\n(n={n})", (etiq_q[i], v),
                     textcoords="offset points", xytext=(8, 5), fontsize=8)

# Matematicas por cuartil
axes[1].bar(etiq_q, p7_seg_pd['mates_prom'],
            color=['#C00000','#ED7D31','#70AD47','#FFC000'], width=0.5)
axes[1].set_ylabel("Puntaje matematicas")
axes[1].set_title("P7: Puntaje matematicas por cuartil de matricula")
axes[1].set_ylim(p7_seg_pd['mates_prom'].min() - 2, p7_seg_pd['mates_prom'].max() + 2)
for i, v in enumerate(p7_seg_pd['mates_prom']):
    axes[1].text(i, v + 0.1, f"{v:.1f}", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---
### P8: ¿Diferencias en desempeño según la organización de la oferta educativa?

**Modelo:** Ninguno, hacer agregacion

In [0]:
# Correlaciones de indicadores de oferta educativa vs puntaje municipal
print("Correlaciones de oferta educativa vs PROM_PUNT_GLOBAL:")
for c, lbl in [
    ('COBERTURA_NETA',    'Cobertura neta'),
    ('COBERTURA_BRUTA',   'Cobertura bruta'),
    ('TASA_MATRICULACION','Tasa matriculacion'),
    ('TOTAL_MATRICULA',   'Total matriculados')
]:
    r = df_mpio.stat.corr(c, 'PROM_PUNT_GLOBAL')
    print(f"  {lbl:25s}: r={r:+.4f}")

# Segmentar por cuartiles de COBERTURA_NETA
q1_cn, q2_cn, q3_cn = df_mpio.approxQuantile('COBERTURA_NETA', [0.25, 0.5, 0.75], 0.01)
print(f"\nCuartiles COBERTURA_NETA: Q1={q1_cn:.1f}  Q2={q2_cn:.1f}  Q3={q3_cn:.1f}")

p8_cob_seg = df_mpio.withColumn('nivel_cobertura',
    F.when(F.col('COBERTURA_NETA') <= q1_cn, '1_Q1 Baja')
     .when(F.col('COBERTURA_NETA') <= q2_cn, '2_Q2')
     .when(F.col('COBERTURA_NETA') <= q3_cn, '3_Q3')
     .otherwise('4_Q4 Alta')
).groupBy('nivel_cobertura').agg(
    F.round(F.mean('COBERTURA_NETA'),1).alias('cob_neta_prom'),
    F.round(F.mean('PROM_PUNT_GLOBAL'),2).alias('puntaje_prom'),
    F.round(F.mean('DESERCION'),2).alias('desercion'),
    F.round(F.mean('REPITENCIA'),2).alias('repitencia'),
    F.count('COD_MUNICIPIO').alias('n_mun')
).orderBy('nivel_cobertura')
print("\nPuntaje por cuartil de cobertura neta:")
p8_cob_seg.show()

# Segmentar por cuartiles de TASA_MATRICULACION
q1_tm, q2_tm, q3_tm = df_mpio.approxQuantile('TASA_MATRICULACION', [0.25, 0.5, 0.75], 0.01)
p8_tm_seg = df_mpio.withColumn('nivel_tasa',
    F.when(F.col('TASA_MATRICULACION') <= q1_tm, '1_Q1 Baja')
     .when(F.col('TASA_MATRICULACION') <= q2_tm, '2_Q2')
     .when(F.col('TASA_MATRICULACION') <= q3_tm, '3_Q3')
     .otherwise('4_Q4 Alta')
).groupBy('nivel_tasa').agg(
    F.round(F.mean('TASA_MATRICULACION'),1).alias('tasa_prom'),
    F.round(F.mean('PROM_PUNT_GLOBAL'),2).alias('puntaje_prom'),
    F.count('COD_MUNICIPIO').alias('n_mun')
).orderBy('nivel_tasa')
print("Puntaje por cuartil de tasa de matriculacion:")
p8_tm_seg.show()

In [0]:
p8_cob_pd  = p8_cob_seg.toPandas().sort_values('nivel_cobertura')
p8_tm_pd   = p8_tm_seg.toPandas().sort_values('nivel_tasa')
p8_scatter = df_mpio.select('COBERTURA_NETA','TASA_MATRICULACION',
                             'PROM_PUNT_GLOBAL').limit(600).toPandas()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
etiq_q4 = ['Q1\nBaja', 'Q2', 'Q3', 'Q4\nAlta']
colores_q = ['#C00000', '#ED7D31', '#FFC000', '#70AD47']

# Scatter: cobertura neta vs puntaje
axes[0].scatter(p8_scatter['COBERTURA_NETA'], p8_scatter['PROM_PUNT_GLOBAL'],
                alpha=0.35, s=14, color='#4472C4')
axes[0].set_xlabel("Cobertura neta (%)")
axes[0].set_ylabel("Puntaje promedio")
axes[0].set_title("P8: Cobertura neta vs Puntaje (muestra n=600)")

# Puntaje por cuartil de cobertura neta
axes[1].bar(etiq_q4, p8_cob_pd['puntaje_prom'], color=colores_q, width=0.6)
axes[1].set_ylabel("Puntaje promedio")
axes[1].set_title("P8: Puntaje por cuartil de cobertura neta")
ymin1 = p8_cob_pd['puntaje_prom'].min() - 3
ymax1 = p8_cob_pd['puntaje_prom'].max() + 3
axes[1].set_ylim(ymin1, ymax1)
for i, (v, n) in enumerate(zip(p8_cob_pd['puntaje_prom'], p8_cob_pd['n_mun'])):
    axes[1].text(i, v + 0.1, f"{v:.1f}\n(n={n})", ha='center', fontsize=9, fontweight='bold')

# Puntaje por cuartil de tasa de matriculacion
axes[2].bar(etiq_q4, p8_tm_pd['puntaje_prom'], color=colores_q, width=0.6)
axes[2].set_ylabel("Puntaje promedio")
axes[2].set_title("P8: Puntaje por cuartil de tasa de matriculacion")
ymin2 = p8_tm_pd['puntaje_prom'].min() - 3
ymax2 = p8_tm_pd['puntaje_prom'].max() + 3
axes[2].set_ylim(ymin2, ymax2)
for i, (v, n) in enumerate(zip(p8_tm_pd['puntaje_prom'], p8_tm_pd['n_mun'])):
    axes[2].text(i, v + 0.1, f"{v:.1f}\n(n={n})", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [0]:
spark.stop()
print("Sesión Spark cerrada.")